# LeetCode 3425 — Longest Special Path bằng Python

- Link: https://leetcode.com/problems/longest-special-path/
- Độ khó: **Hard**
- Chủ đề: **DFS trên cây**, đường gốc → nút như **cửa sổ trượt không lặp giá trị**, **vị trí xuất hiện gần nhất** của mỗi giá trị (có hoàn tác khi quay lui), **tổng tiền tố** độ dài cạnh

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestSpecialPath(self, edges: List[List[int]], nums: List[int]) -> List[int]:
```

## 1. Mục tiêu học

1. Thấy mọi đường đi xuống kết thúc tại `x` là **đoạn cuối** của đường gốc → `x`, nên bài trên cây trở thành bài "đoạn con dài nhất không lặp giá trị" trên một **mảng động**.
2. Duy trì **biên trái** `top` (độ sâu nhỏ nhất được phép) và truyền xuống con như tham số — tự hoàn tác khi quay lui.
3. Duy trì `last[v]` = độ sâu gần nhất của giá trị `v` **trên đường hiện tại**, nhớ **khôi phục** giá trị cũ khi rời nút.
4. Tính độ dài đường bằng tổng tiền tố khoảng cách từ gốc.
5. Tiêu chí phụ: cùng độ dài lớn nhất thì lấy **ít nút nhất**; DFS **không đệ quy** cho cây sâu.

## 2. Đề bài & ví dụ

Cây gốc `0`, cạnh có độ dài; nút `i` có giá trị `nums[i]`. Một **đường đặc biệt** là đường đi **xuống** (tổ tiên → hậu duệ, có thể chỉ một nút) mà mọi giá trị nút **khác nhau**. Trả về `[độ dài lớn nhất, số nút ít nhất trong các đường dài nhất]`.

**Ví dụ 1:** `edges = [[0,1,2],[1,2,3],[1,3,5],[1,4,4],[2,5,6]]`, `nums = [2,1,2,1,3,1]` → `[6, 2]`.

**Ví dụ 2:** `edges = [[1,0,8]]`, `nums = [2,2]` → `[0, 1]`.

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `1 ≤ length ≤ 10^3`, `0 ≤ nums[i] ≤ 5·10^4`.

In [1]:
from typing import List

examples = [
    ([[0, 1, 2], [1, 2, 3], [1, 3, 5], [1, 4, 4], [2, 5, 6]], [2, 1, 2, 1, 3, 1], [6, 2]),
    ([[1, 0, 8]], [2, 2], [0, 1]),
]
for e, a, exp in examples:
    print('n =', len(a), '-> kỳ vọng', exp)

n = 6 -> kỳ vọng [6, 2]
n = 2 -> kỳ vọng [0, 1]


## 3. Kiến thức nền

### 3.1. Cửa sổ trượt "không lặp" trên mảng

Trên mảng `a`, đoạn dài nhất kết thúc tại `i` không có giá trị lặp bắt đầu tại `L_i = max(L_{i−1}, last[a_i] + 1)` với `last[a_i]` là vị trí gần nhất trước đó của `a_i`.

In [2]:
def longest_unique_ending(a):
    last, L, out = {}, 0, []
    for i, x in enumerate(a):
        if x in last:
            L = max(L, last[x] + 1)
        last[x] = i
        out.append(L)
    return out

a = [2, 1, 2, 3, 1]
print('L_i =', longest_unique_ending(a))

L_i = [0, 0, 1, 1, 2]


### 3.2. Từ mảng sang cây

Đường gốc → nút hiện tại là một mảng; đi xuống thì **thêm** cuối, quay lui thì **bỏ** cuối. Biến `L` của cửa sổ trượt phụ thuộc đường, nên truyền nó **xuống con như tham số** (mỗi nhánh có bản riêng). `last[x]` là bảng dùng chung ⇒ khi rời nút phải trả lại giá trị cũ.

### 3.3. Dựng danh sách kề, cha và độ dài từ gốc

In [3]:
def build(n, edges):
    g = [[] for _ in range(n)]
    for u, v, w in edges:
        g[u].append((v, w)); g[v].append((u, w))
    return g

def parents(n, edges):
    g = build(n, edges)
    par, pw, order, seen = [-1] * n, [0] * n, [0], [False] * n
    seen[0] = True
    for u in order:
        for v, w in g[u]:
            if not seen[v]:
                seen[v] = True; par[v], pw[v] = u, w; order.append(v)
    return par, pw, order

print(parents(6, examples[0][0])[:2])

([-1, 0, 1, 1, 1, 2], [0, 2, 3, 5, 4, 6])


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: đường `0 → 1 → 4` có giá trị `2, 1, 3` (khác nhau), độ dài `2 + 4 = 6`, 3 nút. Đường `2 → 5` có giá trị `2, 1`, độ dài `6`, 2 nút. Đường `0 → 1 → 2 → 5` có giá trị `2, 1, 2, 1` — lặp. Đoạn cuối hợp lệ dài nhất kết thúc tại `5` là `2 → 5`.

⇒ Đáp án `[6, 2]` (hai đường cùng dài 6, ít nút nhất là 2).

## 5. Cách 1 — Brute Force: từ mỗi nút leo ngược lên tới khi gặp giá trị lặp

In [4]:
def lsp_brute(edges, nums):
    n = len(nums)
    par, pw, _ = parents(n, edges)
    best = (0, -1)
    for x in range(n):
        seen, length, k, y = set(), 0, 0, x
        while y != -1 and nums[y] not in seen:
            seen.add(nums[y]); k += 1
            best = max(best, (length, -k))
            length += pw[y]; y = par[y]
    return [best[0], -best[1]]

for e, a, exp in examples:
    print(lsp_brute(e, a), '| kỳ vọng', exp)

[6, 2] | kỳ vọng [6, 2]
[0, 1] | kỳ vọng [0, 1]


**Độ phức tạp:** `O(n · depth)` — tệ nhất `O(n²)` trên cây dạng chuỗi giá trị phân biệt.

## 6. Ý tưởng thuật toán / bước đột phá

DFS từ gốc, mỗi phần tử ngăn xếp mang `(nút, cha, độ dài cạnh lên cha, top)`:

- vào nút `u` ở độ sâu `d`: `dist[d] = dist[d − 1] + w`; `old = last.get(nums[u])`; nếu `old` tồn tại thì `top = max(top, old + 1)`; ghi `last[nums[u]] = d`;
- ứng viên: `(dist[d] − dist[top], d − top + 1)`;
- đẩy "rời `u`" (để khôi phục `last[nums[u]] = old`) rồi đẩy các con với `top` mới.

Mỗi nút vào/ra một lần ⇒ `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cửa sổ trượt với tập giá trị, có hoàn tác (đúng nhưng có thể chậm)

Giữ `L` và tập giá trị trong cửa sổ; vào nút thì dịch `L` cho tới khi hết lặp, quay lui thì trả lại các giá trị đã loại. Trên cây "chổi" có thể tốn `O(n · depth)`.

In [5]:
def lsp_window(edges, nums):
    n = len(nums)
    g = build(n, edges)
    vals, dist, inwin = [], [], set()
    L = [0]
    best = [0, 1]
    stack = [(0, -1, 0, False, None)]
    while stack:
        u, p, w, leaving, saved = stack.pop()
        if leaving:
            oldL = saved
            inwin.discard(vals[-1])
            for kk in range(oldL, L[0]):
                inwin.add(vals[kk])
            L[0] = oldL
            vals.pop(); dist.pop()
            continue
        d = len(vals)
        dist.append((dist[-1] if dist else 0) + w)
        vals.append(nums[u])
        oldL = L[0]
        while nums[u] in inwin:
            inwin.discard(vals[L[0]]); L[0] += 1
        inwin.add(nums[u])
        cand = [dist[d] - dist[L[0]], d - L[0] + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, True, oldL))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, False, None))
    return best

for e, a, exp in examples:
    print(lsp_window(e, a), '| kỳ vọng', exp)

[6, 2] | kỳ vọng [6, 2]
[0, 1] | kỳ vọng [0, 1]


### 7.2. Cách 3 — `last` + biên trái `top` truyền xuống (lời giải chính, `O(n)`)

In [6]:
def lsp_top(edges, nums):
    n = len(nums)
    g = build(n, edges)
    last, dist = {}, []
    best = [0, 1]
    stack = [(0, -1, 0, 0, False, None)]          # (nút, cha, w, top, rời?, giá trị last cũ)
    while stack:
        u, p, w, top, leaving, old = stack.pop()
        if leaving:
            if old is None:
                del last[nums[u]]
            else:
                last[nums[u]] = old
            dist.pop()
            continue
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w)
        old = last.get(nums[u])
        if old is not None and old + 1 > top:
            top = old + 1
        last[nums[u]] = d
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, top, True, old))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False, None))
    return best

for e, a, exp in examples:
    print(lsp_top(e, a), '| kỳ vọng', exp)

[6, 2] | kỳ vọng [6, 2]
[0, 1] | kỳ vọng [0, 1]


### 7.3. Cách 4 — Như cách 3 nhưng `last` là **ngăn xếp độ sâu** cho mỗi giá trị

Thay vì lưu "giá trị cũ" để khôi phục, mỗi giá trị giữ một danh sách độ sâu trên đường hiện tại: vào nút thì `append`, rời nút thì `pop`. Phần tử cuối danh sách (trước khi `append`) là lần xuất hiện gần nhất.

In [7]:
from collections import defaultdict

def lsp_stack_lists(edges, nums):
    n = len(nums)
    g = build(n, edges)
    pos = defaultdict(list)
    dist, best = [], [0, 1]
    stack = [(0, -1, 0, 0, False)]
    while stack:
        u, p, w, top, leaving = stack.pop()
        if leaving:
            pos[nums[u]].pop(); dist.pop()
            continue
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w)
        lst = pos[nums[u]]
        if lst and lst[-1] + 1 > top:
            top = lst[-1] + 1
        lst.append(d)
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, top, True))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False))
    return best

for e, a, exp in examples:
    print(lsp_stack_lists(e, a), '| kỳ vọng', exp)

[6, 2] | kỳ vọng [6, 2]
[0, 1] | kỳ vọng [0, 1]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [8]:
def trace(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, path, best = {}, [], [], [0, 1]
    stack = [(0, -1, 0, 0, False, None)]
    while stack:
        u, p, w, top, leaving, old = stack.pop()
        if leaving:
            if old is None: del last[nums[u]]
            else: last[nums[u]] = old
            dist.pop(); path.pop()
            continue
        d = len(dist)
        dist.append((dist[-1] if dist else 0) + w); path.append(u)
        old = last.get(nums[u])
        if old is not None:
            top = max(top, old + 1)
        last[nums[u]] = d
        cand = [dist[d] - dist[top], d - top + 1]
        mark = ''
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best, mark = cand, '  <- tốt hơn'
        print(f"vào {u} (giá trị {nums[u]}), độ sâu {d}, lần trước của giá trị: {old}, top = {top}, "
              f"đoạn {path[top:]} -> {cand}{mark}")
        stack.append((u, p, w, top, True, old))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False, None))
    print('kết quả:', best)

trace(*examples[0][:2])

vào 0 (giá trị 2), độ sâu 0, lần trước của giá trị: None, top = 0, đoạn [0] -> [0, 1]
vào 1 (giá trị 1), độ sâu 1, lần trước của giá trị: None, top = 0, đoạn [0, 1] -> [2, 2]  <- tốt hơn
vào 4 (giá trị 3), độ sâu 2, lần trước của giá trị: None, top = 0, đoạn [0, 1, 4] -> [6, 3]  <- tốt hơn
vào 3 (giá trị 1), độ sâu 2, lần trước của giá trị: 1, top = 2, đoạn [3] -> [0, 1]
vào 2 (giá trị 2), độ sâu 2, lần trước của giá trị: 0, top = 1, đoạn [1, 2] -> [3, 2]
vào 5 (giá trị 1), độ sâu 3, lần trước của giá trị: 1, top = 2, đoạn [2, 5] -> [6, 2]  <- tốt hơn
kết quả: [6, 2]


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def longestSpecialPath(self, edges: List[List[int]], nums: List[int]) -> List[int]:
        n = len(nums)
        g = [[] for _ in range(n)]
        for u, v, w in edges:
            g[u].append((v, w))
            g[v].append((u, w))
        last = [-1] * (max(nums) + 1)            # độ sâu gần nhất của mỗi giá trị trên đường
        dist = []
        best_len, best_cnt = 0, 1
        stack = [(0, -1, 0, 0, False, -1)]
        while stack:
            u, p, w, top, leaving, old = stack.pop()
            if leaving:
                last[nums[u]] = old
                dist.pop()
                continue
            d = len(dist)
            dist.append((dist[-1] if dist else 0) + w)
            old = last[nums[u]]
            if old + 1 > top:
                top = old + 1
            last[nums[u]] = d
            length, cnt = dist[d] - dist[top], d - top + 1
            if length > best_len or (length == best_len and cnt < best_cnt):
                best_len, best_cnt = length, cnt
            stack.append((u, p, w, top, True, old))
            for v, ww in g[u]:
                if v != p:
                    stack.append((v, u, ww, top, False, -1))
        return [best_len, best_cnt]

for e, a, exp in examples:
    got = Solution().longestSpecialPath(e, a)
    print(got, 'OK' if got == exp else 'SAI')

[6, 2] OK
[0, 1] OK


Trong bản nộp, `last` là mảng (giá trị ≤ `5·10^4`) với `−1` nghĩa là "chưa có trên đường"; nhờ vậy `old + 1 = 0` không làm thay đổi `top`.

## 10. Vì sao thuật toán đúng

1. **Đường đặc biệt kết thúc tại `x`** = đoạn `[L, d]` của đường gốc → `x` không lặp giá trị; tính chất "không lặp" đóng với việc thu hẹp từ trái ⇒ có `L` nhỏ nhất, và đó là ứng viên duy nhất cần xét ở `x` (đoạn ngắn hơn có độ dài thực sự nhỏ hơn vì cạnh ≥ 1).
2. **Bất biến của `top`:** `top(x)` = `max` trên mọi nút `y` của đường gốc → `x` của `(lần xuất hiện trước đó của nums[y] trên đường) + 1` — đúng là `L` nhỏ nhất (quy nạp như cửa sổ trượt trên mảng).
3. **`last` đúng:** chỉ chứa độ sâu của các nút trên đường hiện tại, vì khi rời một nút ta khôi phục giá trị trước khi vào.
4. **Tiêu chí phụ** so sánh `(độ dài lớn hơn, số nút ít hơn)`; khởi tạo `[0, 1]` (đường một nút).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Leo ngược | `O(n · depth)` | `O(n)` |
| 2. Cửa sổ trượt có hoàn tác | tệ nhất `O(n · depth)` | `O(n)` |
| 3. `last` + `top` | `O(n)` | `O(n)` |
| 4. Ngăn xếp độ sâu theo giá trị | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'window': lsp_window, 'top': lsp_top, 'stack_lists': lsp_stack_lists,
         'submit': lambda e, a: Solution().longestSpecialPath(e, a)}
edge = [
    ([[0, 1, 5]], [1, 2], [5, 2]),
    ([[0, 1, 1], [1, 2, 1]], [7, 7, 7], [0, 1]),
    ([[0, 1, 3], [0, 2, 3]], [1, 2, 3], [3, 2]),
    ([[0, 1, 1], [1, 2, 10], [2, 3, 1]], [1, 2, 1, 3], [11, 3]),
]
for e, a, exp in examples + edge:
    assert lsp_brute(e, a) == exp, (e, a, lsp_brute(e, a))
    for name, f in impls.items():
        assert f(e, a) == exp, (name, e, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random

def random_tree(n, vmax, wmax, rng):
    edges = [[rng.randrange(v), v, rng.randint(1, wmax)] for v in range(1, n)]
    perm = list(range(1, n)); rng.shuffle(perm); perm = [0] + perm
    return ([[perm[u], perm[v], w] if rng.random() < 0.5 else [perm[v], perm[u], w] for u, v, w in edges],
            [rng.randint(0, vmax) for _ in range(n)])

rng = random.Random(3425)
cnt = 0
for _ in range(1500):
    e, a = random_tree(rng.randint(2, 14), rng.randint(0, 5), rng.choice([1, 3, 10]), rng)
    exp = lsp_brute(e, a)
    for name, f in impls.items():
        assert f(e, a) == exp, (name, e, a)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 1500 cây ngẫu nhiên.


### 12.3. Ca lớn `n = 5·10^4`

In [12]:
import time
n = 5 * 10 ** 4
rng = random.Random(1)
chain = ([[i, i + 1, rng.randint(1, 1000)] for i in range(n - 1)], list(range(n)))
star = ([[0, i, rng.randint(1, 1000)] for i in range(1, n)], [rng.randint(0, 10) for _ in range(n)])
rand_big = random_tree(n, 50, 1000, rng)
for name, (e, a) in [('chuỗi giá trị khác nhau', chain), ('sao', star), ('ngẫu nhiên', rand_big)]:
    t = time.time(); r = Solution().longestSpecialPath(e, a); t1 = time.time() - t
    t = time.time(); r2 = lsp_top(e, a); t2 = time.time() - t
    assert r == r2
    print(f"{name:24s}: {r} | nộp bài {t1:.2f}s | cách 3 {t2:.2f}s")

chuỗi giá trị khác nhau : [24938555, 50000] | nộp bài 0.04s | cách 3 0.09s
sao                     : [1000, 2] | nộp bài 0.08s | cách 3 0.03s


ngẫu nhiên              : [10015, 15] | nộp bài 0.14s | cách 3 0.16s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [13]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        e, a = random_tree(rng.randint(2, 8), 3, 4, rng)
        if wrong(e, a) != lsp_brute(e, a) and (best is None or len(a) < len(best[1])):
            best = (e, a)
    return best

### Lỗi 1 — Không khôi phục `last` khi rời nút (giá trị ở nhánh anh em "rò rỉ")

In [14]:
def wrong_no_restore(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, best = {}, [], [0, 1]
    stack = [(0, -1, 0, 0, False)]
    while stack:
        u, p, w, top, leaving = stack.pop()
        if leaving:
            dist.pop(); continue                     # SAI: không trả last về giá trị cũ
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        old = last.get(nums[u])
        if old is not None and old < d:
            top = max(top, old + 1)
        last[nums[u]] = d
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, top, True))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False))
    return best

ce = find_counterexample(wrong_no_restore, seed=1)
print('phản ví dụ:', ce, '| sai:', wrong_no_restore(*ce), '| đúng:', lsp_brute(*ce))

phản ví dụ: ([[0, 2, 2], [1, 0, 3]], [0, 0, 0]) | sai: [2, 2] | đúng: [0, 1]


**Vì sao sai:** sau khi rời một nhánh, `last` vẫn nhớ độ sâu của nút ở nhánh đó; nhánh khác cùng độ sâu bị coi là "đã có giá trị này trên đường".

### Lỗi 2 — Chỉ dùng `last[nums[u]] + 1`, quên lấy `max` với `top` của cha

In [15]:
def wrong_no_max(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, best = {}, [], [0, 1]
    stack = [(0, -1, 0, False, None)]
    while stack:
        u, p, w, leaving, old = stack.pop()
        if leaving:
            if old is None: del last[nums[u]]
            else: last[nums[u]] = old
            dist.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        old = last.get(nums[u])
        top = old + 1 if old is not None else 0       # SAI: bỏ mất ràng buộc từ tổ tiên
        last[nums[u]] = d
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, True, old))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, False, None))
    return best

ce = find_counterexample(wrong_no_max, seed=2)
print('phản ví dụ:', ce, '| sai:', wrong_no_max(*ce), '| đúng:', lsp_brute(*ce))

phản ví dụ: ([[2, 0, 1], [1, 2, 3]], [2, 1, 2]) | sai: [4, 3] | đúng: [3, 2]


**Vì sao sai:** một cặp lặp **ở phía trên** (giữa hai tổ tiên) vẫn làm đoạn không hợp lệ dù giá trị của nút hiện tại chưa từng xuất hiện.

### Lỗi 3 — Hòa độ dài thì không xét số nút

In [16]:
def wrong_tie(edges, nums):
    r = lsp_top(edges, nums)
    n = len(nums); par, pw, _ = parents(n, edges)
    best = (0, 0)
    for x in range(n):
        seen, length, k, y = set(), 0, 0, x
        while y != -1 and nums[y] not in seen:
            seen.add(nums[y]); k += 1
            if length > best[0]:                     # SAI: không cập nhật khi hòa với ít nút hơn
                best = (length, k)
            length += pw[y]; y = par[y]
    return [best[0], best[1] if best[1] else 1]

e, a, _ = examples[0]
print('sai :', wrong_tie(e, a), '| đúng:', Solution().longestSpecialPath(e, a))

sai : [6, 3] | đúng: [6, 2]


**Vì sao sai:** ví dụ 1 có hai đường dài 6 (`0 → 1 → 4` với 3 nút gặp trước, `2 → 5` với 2 nút). Phải chọn **ít nút nhất**.

### Lỗi 4 — Đệ quy sâu với giới hạn mặc định

In [17]:
def naive_depth(edges, nums):
    g = build(len(nums), edges)
    def dfs(u, p):
        return 1 + max((dfs(v, u) for v, _ in g[u] if v != p), default=0)
    return dfs(0, -1)

try:
    naive_depth(*chain)
except RecursionError as ex:
    print('RecursionError:', ex)

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** cây dạng chuỗi `5·10^4` nút vượt giới hạn đệ quy mặc định (1000). Dùng ngăn xếp tự quản.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi xuống, điều kiện kiểu cửa sổ" ⇒ cửa sổ trượt trên đường gốc → nút.
- Thông tin **truyền xuống** (biên trái) đi theo tham số; thông tin **dùng chung** (`last`) phải hoàn tác.
- Độ dài đường = hiệu tiền tố khoảng cách từ gốc.
- Giá trị nhỏ (≤ `5·10^4`) ⇒ dùng mảng thay dict cho nhanh.

## 15. Biến thể của bài

1. **LC 3486 — bản II:** cho phép **một** giá trị lặp đúng hai lần ⇒ giữ **hai** biên lớn nhất.
2. **LC 3 — Longest Substring Without Repeating Characters:** cùng ý tưởng trên mảng.
3. Đếm số đường đặc biệt ⇒ cộng `d − top + 1` tại mỗi nút.
4. Đường bất kỳ (không nhất thiết đi xuống) ⇒ khó hơn nhiều (centroid decomposition…).

In [18]:
# Biến thể 3: đếm số đường đặc biệt (kể cả đường một nút)
def count_special(edges, nums):
    n = len(nums); g = build(n, edges)
    last, depth, total = {}, 0, 0
    stack = [(0, -1, 0, False, None)]
    cur_depth = [0]
    while stack:
        u, p, top, leaving, old = stack.pop()
        if leaving:
            if old is None: del last[nums[u]]
            else: last[nums[u]] = old
            cur_depth[0] -= 1; continue
        d = cur_depth[0]; cur_depth[0] += 1
        old = last.get(nums[u])
        if old is not None:
            top = max(top, old + 1)
        last[nums[u]] = d
        total += d - top + 1
        stack.append((u, p, top, True, old))
        for v, _ in g[u]:
            if v != p:
                stack.append((v, u, top, False, None))
    return total

def count_brute(edges, nums):
    n = len(nums); par, _, _ = parents(n, edges); total = 0
    for x in range(n):
        seen, y = set(), x
        while y != -1 and nums[y] not in seen:
            seen.add(nums[y]); total += 1; y = par[y]
    return total

rng = random.Random(15)
for _ in range(400):
    e, a = random_tree(rng.randint(2, 12), 3, 5, rng)
    assert count_special(e, a) == count_brute(e, a)
print('Biến thể đếm khớp vét cạn; ví dụ 1 có', count_special(*examples[0][:2]), 'đường đặc biệt.')

Biến thể đếm khớp vét cạn; ví dụ 1 có 11 đường đặc biệt.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Leo ngược + tập | `O(n · depth)` | Oracle |
| 2 | Cửa sổ trượt có hoàn tác | tệ nhất `O(n · depth)` | Trực quan |
| 3 | `last` + `top` | `O(n)` | **Ý tưởng chính** |
| 4 | Ngăn xếp độ sâu theo giá trị | `O(n)` | Hoàn tác tự nhiên |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** chuỗi `0 – 1 – 2` cạnh dài 1, `nums = [5, 6, 5]`.

**Bài 2 (tính tay):** sao tâm `0` với 3 lá, cạnh dài `1, 2, 3`, `nums = [1, 1, 2, 3]`.

**Bài 3 (điền TODO):** viết `longest_unique_substring(s)` (LC 3) bằng cùng kỹ thuật.

**Bài 4 (tự viết lại):** trả về **chính đường đi** (danh sách nút) của một đường tối ưu.

**Bài 5 (mini challenge):** biến thể "giá trị liên tiếp không được bằng nhau" (chỉ cấm hai nút **kề nhau** cùng giá trị).

In [19]:
def longest_unique_substring_todo(s):
    # TODO
    raise NotImplementedError

def best_path_nodes_todo(edges, nums):
    # TODO
    raise NotImplementedError

def no_adjacent_equal_todo(edges, nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Cả đường lặp giá trị 5. (2) Đoạn `0 → 1` và `1 → 2` đều dài 1. (3) `[1, 2]`.

**Bài 2.** (1) Đường từ lá lên tâm: lá 1 cùng giá trị với tâm. (2) Lá 2, 3 hợp lệ với tâm. (3) `[3, 2]`.

**Bài 3.** (1) `last` theo ký tự. (2) `L = max(L, last + 1)`. (3) Đối chiếu ví dụ LC 3.

**Bài 4.** (1) Ghi lại `(top, d, nút)` của ứng viên tốt nhất cùng đường hiện tại. (2) Lưu bản sao đường khi cập nhật. (3) Kiểm tra giá trị khác nhau và độ dài khớp.

**Bài 5.** (1) Chỉ cần so với giá trị **cha**. (2) `top = d` nếu `nums[u] == nums[cha]`. (3) Đối chiếu vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().longestSpecialPath([[0, 1, 1], [1, 2, 1]], [5, 6, 5])
a2 = Solution().longestSpecialPath([[0, 1, 1], [0, 2, 2], [0, 3, 3]], [1, 1, 2, 3])
assert a1 == lsp_brute([[0, 1, 1], [1, 2, 1]], [5, 6, 5]) == [1, 2]
assert a2 == lsp_brute([[0, 1, 1], [0, 2, 2], [0, 3, 3]], [1, 1, 2, 3]) == [3, 2]
print('Bài 1:', a1, '| Bài 2:', a2)

def longest_unique_substring(s):
    last, L, best = {}, 0, 0
    for i, ch in enumerate(s):
        if ch in last:
            L = max(L, last[ch] + 1)
        last[ch] = i
        best = max(best, i - L + 1)
    return best

assert [longest_unique_substring(s) for s in ["abcabcbb", "bbbbb", "pwwkew", ""]] == [3, 1, 3, 0]
print('Bài 3: LC 3 ví dụ ->', [longest_unique_substring(s) for s in ["abcabcbb", "bbbbb", "pwwkew"]])

Bài 1: [1, 2] | Bài 2: [3, 2]
Bài 3: LC 3 ví dụ -> [3, 1, 3]


In [21]:
# Bài 4, 5
def best_path_nodes(edges, nums):
    n = len(nums); g = build(n, edges)
    last, dist, path = {}, [], []
    best, best_path = [0, 1], [0]
    stack = [(0, -1, 0, 0, False, None)]
    while stack:
        u, p, w, top, leaving, old = stack.pop()
        if leaving:
            if old is None: del last[nums[u]]
            else: last[nums[u]] = old
            dist.pop(); path.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w); path.append(u)
        old = last.get(nums[u])
        if old is not None:
            top = max(top, old + 1)
        last[nums[u]] = d
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best, best_path = cand, path[top:]
        stack.append((u, p, w, top, True, old))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False, None))
    return best, best_path

def no_adjacent_equal(edges, nums):
    n = len(nums); g = build(n, edges)
    dist, best = [], [0, 1]
    stack = [(0, -1, 0, 0, False)]
    while stack:
        u, p, w, top, leaving = stack.pop()
        if leaving:
            dist.pop(); continue
        d = len(dist); dist.append((dist[-1] if dist else 0) + w)
        if p != -1 and nums[p] == nums[u]:
            top = d
        cand = [dist[d] - dist[top], d - top + 1]
        if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
            best = cand
        stack.append((u, p, w, top, True))
        for v, ww in g[u]:
            if v != p:
                stack.append((v, u, ww, top, False))
    return best

def no_adj_brute(edges, nums):
    n = len(nums); par, pw, _ = parents(n, edges); best = (0, -1)
    for x in range(n):
        length, k, y = 0, 1, x
        best = max(best, (0, -1))
        while par[y] != -1 and nums[par[y]] != nums[y]:
            length += pw[y]; k += 1; y = par[y]
            best = max(best, (length, -k))
    return [best[0], -best[1]]

rng = random.Random(19)
par_cache = {}
for _ in range(400):
    e, a = random_tree(rng.randint(2, 12), 3, 5, rng)
    (L, c), nodes = best_path_nodes(e, a)
    par, pw, _ = parents(len(a), e)
    assert [L, c] == lsp_brute(e, a) and len(nodes) == c and len({a[v] for v in nodes}) == c
    assert sum(pw[v] for v in nodes[1:]) == L and all(par[nodes[i + 1]] == nodes[i] for i in range(c - 1))
    assert no_adjacent_equal(e, a) == no_adj_brute(e, a)
print('Bài 4, 5 khớp vét cạn; ví dụ 1:', best_path_nodes(*examples[0][:2]))

Bài 4, 5 khớp vét cạn; ví dụ 1: ([6, 2], [2, 5])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
stack = [(0, -1, 0, top=0, leaving=False, old)]
vào u ở độ sâu d: dist[d] = dist[d-1] + w
    old = last[nums[u]];  top = max(top, old + 1);  last[nums[u]] = d
    cập nhật (dist[d] - dist[top], d - top + 1)
    đẩy (u, rời, old); đẩy các con với top
rời u: last[nums[u]] = old; dist.pop()
```

**Checklist:**

- [ ] Khôi phục `last` khi rời nút?
- [ ] `top = max(top_cha, old + 1)`?
- [ ] Hòa độ dài ⇒ ít nút hơn; khởi tạo `[0, 1]`?
- [ ] DFS lặp (không đệ quy sâu)?
- [ ] Độ dài đoạn bằng hiệu tiền tố `dist[d] − dist[top]`?